# EEE3027 Lab 3: Implementation of Producer-Consumer-based Applications

In this lab, you will learn how to:

1. Create concurrent processes.
2. Create thread safe locking variables including mutexes.

## What is a Producer-Consumer pattern

The Producer-Consumer pattern is a common design paradigm used in distributed systems, where multiple components are required to communicate with one another to perform a task. 

The core key idea is that you have two types of components that are **_producers_** and **_consumers_**. The former 'produces' the data/events and adds them to the shared memory location, while the latter retrieves the data/events from the shared memory location and 'consumes' them. This is illustrated using the example below.

<center><img src="./images/producer_consumer.png" width=500px height=500px /></center>

## Benefits

- Scalable: The throughput of the system can be easily increased by (carefully) adding more producers or consumers.
- Fault tolerant: If one producer or consumer has a fault, it can be easily replaced/restarted without affecting the overall system.
- Load handling: When the data arrives in 'bursts', the queue acts as a buffer where it allows producers to dump data at high speeds without overwhelming consumers (which may be working at a slower speed). 

## Detriments

- Incresed complexity: By introducing a shared memory component, one must manage thread synchronisation, handle potential deadlocks and monitor the health of the buffer.
- Latency overhead: Because data is serialised, 'produced' to the queue and then 'consumed' from the queue, this inherently adds latency that may not suitable for low-latency systems, where every microsecond counts (e.g. an airplane's safety mechanism). 
- Resource management risks: Due to producers being significantly faster than their consumer counterparts, one must carefully decide the type of queue they implement. For example, on one hand, if the queue is unbounded then there is a risk of memory overflow. On the other hand, if the queue is bounded, the procuders are either blocked (i.e. deadlocked) or drops data (i.e. data loss). 

## Real-world Example: Co-Simulation of an IoT Sensor and IoT Processor

> **_Note_**: This example below aims to simulate how sensing (and data conditioning) followed by processing data can be implemented as two concurrent processes. Between these two processes, they share a circular buffer/memory among others, where it is assumed to ensure that the sensed data are stored and processed without any potential hazards.

Sensors in an IoT system continuously collect real-world data such as temperature, pressure, motion, or humidity. These sensors generate raw data in the form of electrical signals, which are then converted into digital values using an **Analog-to-Digital Converter (ADC)**. In this example, we will assume the details of ADC are masked and we are able to generate random sensing data for simulation purposes. 

The sensor data is stored in a **shared memory** (in this case a circular buffer with a capacity of 10 data samples max). In our case, we will assume two sensors are writing data to this shared memory using synchronization mechanisms to prevent data corruption.  

An **IoT processor** (which could be a microcontroller, edge device, or cloud server agent) retrieves this stored data, processes it (for example for filtering, anomaly detection, or machine learning models, and then interprets) for decision-making. This processed information is either **displayed to users**, **triggering an action**, or **transmitted to cloud services** for further analysis. In our example, we will just assume each data are arithmetically processed for data conditioning.

This seamless flow from data generation to interpretation enables **real-time monitoring and automation** in IoT applications such as smart homes, healthcare, and industrial automation. 🚀

Using the diagram above, the 'Sensor' is our 'Producer' and the 'Processor' is our 'Consumer'

## Code Implementation

> **_Note_**: This example assumes that the designer has already taken the necessary measures to prevent the producer-consumer issues described above. Your tasks further down in the Notebook will not have this luxury, as you will need to first manage how the system should handle the data without such protocols (yet).

In [2]:
import threading
import time
import random

random.seed(42) ### Apply random seed to ensure reproducibility

In [ ]:
# Producer Thread Class
class Sensor(threading.Thread): # Sensor is creating the data, hence the producer

    def run(self): # defined the run method
     
        global CAPACITY, buffer, in_index, out_index # Global variables that show the factors for the buffer and its capacity, as well as the indices for the producer and consumer
        global mutex, empty, full # for mutex and semaphores
         
        items_produced = 0 # initial state, no data produced yet
        
        while items_produced < CAPACITY: # Temporary condition so the Producer does not run forever more
            empty.acquire() # adheres to FIFO structure
            mutex.acquire() # need to read

            data = random.randint(1, 100) # data randomly generated between 1 and 100
            buffer[in_index] = data # storing the data in the buffer at the index of the input or producer
            print(f"Sensor {self.name} sensed data '{data}' and stored in memory location {in_index}. Current state of buffer: {buffer}")
            in_index = (in_index + 1) % CAPACITY # iterates to next position, then uses modulo to wrap around to the beginning of the buffer if it reaches the end
            items_produced += 1 # add 1 each time data is produced.
            
            mutex.release() # thread method that I haven't read up on yet.
            full.release()
            
            time.sleep(1) # Sleep for a while to simulate time taken to produce data

In [ ]:
# Consumer Thread Class
class Processor(threading.Thread): # processor takes in the data, hence the consumer
    
    def run(self):
     
        global CAPACITY, buffer, in_index, out_index, counter # same variables as before and a counter added in.
        global mutex, empty, full
         
        items_consumed = 0 # 
        while items_consumed < CAPACITY: # Similarly, temporary condition so the Consumer does not run forever more
            full.acquire() # when it is ready, it will get the full semaphore, which means there is data to consume
            mutex.acquire()
            
            data = buffer[out_index] # read operation, getting the data from the buffer at the output
            buffer[out_index] = None # clear the buffer at that index after reading
            print(f"Processor {self.name} processed '{data}' from memory location {out_index + 1}. Current state of buffer: {buffer}")
            out_index = (out_index + 1) % CAPACITY
            items_consumed += 1
            
            mutex.release()
            empty.release()      
            
            time.sleep(2.5) # This is the faster thread so it has to sleep for a longer time to simulate the time taken to process data and to allow the producer to produce more data.
            
            

In [5]:
# Shared Memory variables
CAPACITY = 5
buffer = [None for _ in range(CAPACITY)]

in_index = 0
out_index = 0
 
# Declaring Mutexes and thread-safe signals
mutex = threading.Semaphore()
empty = threading.Semaphore(CAPACITY)
full = threading.Semaphore(0)
 
# Creating Threads
sensor = Sensor()
processor = Processor()
 
# Starting Threads
sensor.start()
processor.start()
 
# Waiting for threads to complete
sensor.join()
processor.join()

Sensor Thread-4 sensed data '82' and stored in memory location 0. Current state of buffer: [82, None, None, None, None]
Processor Thread-5 processed '82' from memory location 1. Current state of buffer: [None, None, None, None, None]
Sensor Thread-4 sensed data '15' and stored in memory location 1. Current state of buffer: [None, 15, None, None, None]
Sensor Thread-4 sensed data '4' and stored in memory location 2. Current state of buffer: [None, 15, 4, None, None]
Processor Thread-5 processed '15' from memory location 2. Current state of buffer: [None, None, 4, None, None]
Sensor Thread-4 sensed data '95' and stored in memory location 3. Current state of buffer: [None, None, 4, 95, None]
Sensor Thread-4 sensed data '36' and stored in memory location 4. Current state of buffer: [None, None, 4, 95, 36]
Processor Thread-5 processed '4' from memory location 3. Current state of buffer: [None, None, None, 95, 36]
Processor Thread-5 processed '95' from memory location 4. Current state of buf

# Your Tasks

### Firstly, initialise these simple Producer and Consumer classes

In [ ]:
class SimpleProducer(threading.Thread):
    def run(self):
        global CAPACITY, buffer
        in_index = 0
        while in_index < CAPACITY:
            value = random.randint(1, 100)
            buffer[in_index] = value
            in_index = (in_index + 1)
            print(f"Producer {self.name} produced '{value}' at buffer[{in_index}].")
            time.sleep(1)

class SimpleConsumer(threading.Thread):
    def run(self):
        global CAPACITY, buffer
        out_index = 0
        while out_index < CAPACITY :
            item = buffer[out_index]
            buffer[out_index] = None ## 'Consumed'
            out_index = (out_index + 1)        
            print(f"Consumer {self.name} consumed '{item}' from buffer[{out_index}].")
            time.sleep(2)

### Use the below baseline code below to complete your tasks

In [7]:
# Shared Memory variables
CAPACITY = 5
buffer = [None for i in range(CAPACITY)]
 
# Creating Threads
producer = SimpleProducer()
consumer = SimpleConsumer()
 
# Starting Threads
producer.start()
consumer.start()
 
# Waiting for threads to complete
producer.join()
consumer.join()

Producer Thread-6 produced '32' at buffer[1].
Consumer Thread-7 consumed '32' from buffer[1].
Producer Thread-6 produced '29' at buffer[2].
Producer Thread-6 produced '18' at buffer[3].
Consumer Thread-7 consumed '29' from buffer[2].
Producer Thread-6 produced '95' at buffer[4].
Producer Thread-6 produced '14' at buffer[5].
Consumer Thread-7 consumed '18' from buffer[3].
Consumer Thread-7 consumed '95' from buffer[4].
Consumer Thread-7 consumed '14' from buffer[5].


### 1. Copy the baseline code and change the buffer's capacity. Does it still work as expected?

In [8]:
# Insert your code here
# Shared Memory variables
CAPACITY = 7
buffer = [None for i in range(CAPACITY)]
 
# Creating Threads
producer = SimpleProducer()
consumer = SimpleConsumer()
 
# Starting Threads
producer.start()
consumer.start()
 
# Waiting for threads to complete
producer.join()
consumer.join()

Producer Thread-8 produced '87' at buffer[1].
Consumer Thread-9 consumed '87' from buffer[1].
Producer Thread-8 produced '95' at buffer[2].
Producer Thread-8 produced '70' at buffer[3].
Consumer Thread-9 consumed '95' from buffer[2].
Producer Thread-8 produced '12' at buffer[4].
Producer Thread-8 produced '76' at buffer[5].
Consumer Thread-9 consumed '70' from buffer[3].
Producer Thread-8 produced '55' at buffer[6].
Producer Thread-8 produced '5' at buffer[7].
Consumer Thread-9 consumed '12' from buffer[4].
Consumer Thread-9 consumed '76' from buffer[5].
Consumer Thread-9 consumed '55' from buffer[6].
Consumer Thread-9 consumed '5' from buffer[7].


### 2. Copy the baseline code (or your increased capacity implementation) and increase the number of producers. What happens? 

In [9]:
# Insert your code here
# Shared Memory variables
CAPACITY = 5
buffer = [None for i in range(CAPACITY)]
 
# Creating Threads
producer = SimpleProducer()
producer1= SimpleProducer()
consumer = SimpleConsumer()
 
# Starting Threads
producer.start()
producer1.start()
consumer.start()
 
# Waiting for threads to complete
producer.join()
producer1.join()
consumer.join()

Producer Thread-22 produced '12' at buffer[1].
Consumer Thread-23 consumed '12' from buffer[1].
Producer Thread-22 produced '30' at buffer[2].
Producer Thread-22 produced '78' at buffer[3].
Consumer Thread-23 consumed '30' from buffer[2].
Producer Thread-22 produced '72' at buffer[4].
Consumer Thread-23 consumed '78' from buffer[3].
Producer Thread-22 produced '92' at buffer[5].
Consumer Thread-23 consumed '72' from buffer[4].
Consumer Thread-23 consumed '92' from buffer[5].


### 3. Copy the baseline code (or your increased capacity implementation) and increase the number of consumers. What happens?

In [ ]:
# Insert your code here

### 4. Given what you know from Tasks 2 and 3: Copy the baseline code (or your increased capacity implementation) and increase both the numbers of producers and consumers. Make sure you prevent the situations from Tasks 2 and 3 accordingly.

In [ ]:
# Insert your code here

### 5. Implement a new function that can measure the latency of the baseline code. How can you use this to determine the speed of the producers and consumers?

In [ ]:
# Insert your code here

### 6. Implement a FIFO implementation for your own scenario using the code(s) above. What applications can benefit from such a data structure?

In [ ]:
# Insert your code here

# [Optional] Further Learning (Theory): Conceptualising the Producer-Consumer FIFOs as Petri nets

The following section contains the theory for designing these 'FIFOs' using Petri nets.

> **_Note_**: This year, you are not expected to design any Petri nets. The materials below were inherited from last year's work and only serve as additional learning material on how you can conceptualise these buffers using so-called '1-bounded' (i.e. 1-safe) Petri nets or unbounded Petri nets.

## What is a Petri net

Petri nets are a mathematical model (like Finite State Machines) that can be used to specify and analyze many systems including distributed systems and asynchronous circuits. They typically contain a finite set of **places**, **transitions**, **arcs**, where they contain a **marking** (i.e. places that contain a **token**) that depicts the system's state. Transitions are typically labelled with "events" and can only **fire** if all of its preceding places contain a token, where upon firing, the tokens are "consumed" from all preceding places and all tokens are "produced" to all suceeding places.

The diagram below shows a simple comparison between the C-element gate, the Petri net and the Signal Transition Graph (which is just a circuit Petri net). 

<center><img src="./images/c-element.png" width=90% /></center>

## Representing the FIFO buffers of the Sensor-Processor Co-simulation example as a Petri net

Below we generated two (high-level) Petri Net representations of this example: the first model assumes an **unsafe model** and the second model assumes a **1-safe model** maximally. For each model, we will discuss their design choices with respect to the model complexity and safety tradeoffs. 

> **_Note:_** Our Petri Net models will be based on the Python code implementation shown below.
>
> Also, to help reduce the size of our 1-safe Petri Net, we will only model up to three memories and make use of Workcraft's features, e.g. *proxy places*. 

### Unsafe Petri Net Design

<center><img src="./images/unsafe-petri-net.png" width="50%" /></center>

In our unsafe Petri Net model, we have:

- **Seven transitions**. Three transitions called *sensor1*, *sensor2* and *processor1* that each represent 'Sensor 1', 'Sensor 2' and 'Processor' respectively. Another three transitions, where two transitions called *put_1* and *put_2* represent when the sensors add data to the buffer/memory and one transition called *get* represent when the processor retrieves data from the buffer/memory. Lastly, we also have an additional transition called *reset_mux* that replenishes the token back at place *mux* to reset the system for the next call. 

- **Ten places**. Three places that represent the semaphores used in the following implementation called *mux* (representing our mutex), *empty* (representing when there is data to be written), and *full* (representing when there is data to be retrieved. Another three places called *sens1_do_op*, *sens2_do_op* and *proc1_do_op* that represent when *sensor1*, *sensor2* and *processor1* are in operation after receiving the token from *mux* respectively. Three more places called *sens1_cnt*, *sens2_cnt* and *proc1_cnt* used to count the number of items handled by *sensor1*, *sensor2* and *processor1* respectively. Finally, one place called *op_done* used to reset the system via *reset_mux*.

> **_Note:_** As *mux*, *empty* and *full* implement the three semaphores in our implementation, it is important to take note of the numbers assigned to each of them as these will act as our **tokens**. In this case, *mux* will have one token but *empty* will have 10 tokens and *full* will have 0 tokens.

To capture the behaviour of the implementation below, we:

- First connect *mux* to *sensor1*, *sensor2* and *processor1*, as well as from *sensor1*, *sensor2* and *processor1* to their respectively *do_op* and *cnt* places.
  
- We also connect the *put* transitions and *get* transition to form the basic producer-consumer Petri Net (with one exception being that we have two put transitions and 10 tokens instead of 1).

- To allow our system to reset, a connection is made from the *put* and *get* transitions to the place *op_done* before we connect an arc from it to *reset_mux* and from *reset_mux* back to *mux*.

- Lastly, we also connect read arcs between the *put* transitions and *empty* place to ensure the sensors can only add data when there is empty space, as well as between the *get* transition and *full* transition to ensure the processor can only retrieve data when there is occupied space. 

> **_Note:_** The final unsafe Petri Net design is very compact and easily captures the behaviour of our Python implementation below. However, despite this, it is worth mentioning that the unsafe Petri Net gives no indicator of what order the data is being handled in -- If you carefully examine the Python code, you will realise that data is actually handled sequentially in a round-robin order.
>
> But, we can, of course, simply assume that the data will be handled in the order that it has been placed by the sensors. **However, do mind that you must explicitly state this when you do such an approach**.  
>
> As *mux*, *empty* and *full* implement the three semaphores in our implementation, it is important to take note of the numbers assigned to each of them as these will act as our **tokens**. In this case, *mux* will have one token but *empty* will have 10 tokens and *full* will have 0 tokens.

### 1-safe Petri Net Design

<center><img src="./images/safe-petri-net.png" width="95%" /></center>

In our 1-safe Petri Net model, we have:

- **Nineteen transitions**. Three dedicated transitions of each 'Sensor 1', 'Sensor 2' and 'Processor' called *sensor1*, *sensor2* and *processor1* followed by an assigned number from 1 to 3 (totalling Nine transitions). Three *put_..._1*, *put_..._2* and *get_...* transitions for each of the three memories (also totalling Nine transitions). Lastly, like the Unsafe Petri Net, an additional transition called *reset_mux* that replenishes the token back at place *mux* to reset the system for the next call. 

- **Seventeen places**. Three dedicated places of each *empty* and *full* semaphore for each of the three memories (totalling Six places) followed by one place representing the *mux*. Three places for each of the *put* and *get* sides to form a **queue** between the memroies (totalling Six places). Another three places called *sens1_do_op*, *sens2_do_op* and *proc1_do_op*, as well another place called *op_done* that have the same behaviour as their counterpart in the Unsafe Petri Net design. 

> **_Note:_** As the number of memories are explicitated, the number of *empty* and *full* places depend on the number of memories there are in the design, e.g. if you have 20 memories then you must have 20 *empty* places and 20 *full* places, with a token assigned to each *empty* place initially. 

To capture the behaviour of the implementation below, we:

- Each *sensor1*, *sensor2* and *processor1* transition must be connected by a producing arc from place *mux* and by a consuming arc to their respective *do_op* place.

- Additionally, transitions *sensor1*, *sensor2* and *processor1* must each be connected with a read arc between it and their corresponding memory's empty/full place (e.g. *sensor1/2_1* connects to *empty01*, *processor1_1* to *full01*, *sensor1/2_2* to *empty02*, etc.), and a read arc between it and their corresponding memory's queueing place (e.g. *sensor1/2_1* connects to *p1*, *processor1_1* to *g1*, *sensor1/2_2* to *p2*, etc.). This is to ensure the sensors can only add data when there is empty space and the processors can only retrieve data when there is occupied space. 

- Each of the *do_op* places connect to their respective *put/get* transitions.

- Each memory, their pair of *put* transitions and *get* transition is connected with their empty and full places to form the basic producer-consumer Petri net segment.

- Every *put* and *get* transition is connected to the *op_done* place, followed by a connection from *op_done* to *reset_mux* and from *reset_mux* to *mux* to allow the system to reset.

> **_Note:_** Unlike our unsafe Petri Net design, our 1-safe Petri Net design is much larger even when only considering three memories (meaning if we do all ten memories or even more, our design can exponentially grow!).
>
> However, the advantages over the unsafe Petri net is that this model can be easily verified and implemented on to Asynchronous platforms, and we can see a near 1-to-1 behaviour between it and the Python implementation and the order of how data is handled.
>
> Unfortunately (unless you do some very clever modelling), there is not an easier way to make a very clean Safe Petri net -- This is especially the case if you want to **explictate the memories being occupied/taken in accordance with the sensors/processor**.